# Paso 1: Recuperar y verificar los componentes del avance de proyecto.

## Tabla de trazabilidad — Avance de Proyecto

| Elemento | Archivo o ubicación | Versión | Estado | Acción requerida |
|---|---|---|---|---|
| Dataset | Se carga vía `kagglehub` (`blastchar/telco-customer-churn`), no hay CSV local en el repo | Consultado 25–31 ago 2026 | Disponible (requiere `kagglehub` y conexión) | Documentar que requiere autenticación/conexión a Kaggle |
| Pipeline | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 13) | ColumnTransformer con imputación (mediana), StandardScaler, OneHotEncoder, OrdinalEncoder | Corre limpio | Ninguna |
| Baseline | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 14) | DummyClassifier (most_frequent) — Acc 0.7346, Recall 0.0000 | Completo | Ninguna |
| Modelo preliminar | `Avance_Proyecto/Avance_Proyecto.ipynb` (Pasos 15–19) | Regresión Logística (max_iter=1000, solver='lbfgs') — Acc 0.8055, F1 0.6040, ROC-AUC 0.8421 | Seleccionado y justificado | Ninguna |
| Métricas | `Avance_Proyecto/Avance_Proyecto.ipynb` (Paso 16) | Accuracy, Precision, Recall, F1, ROC-AUC para ambos modelos | Completo | Ninguna |
| Informe del avance de proyecto | `Avance_Proyecto/Avance_Proyecto.ipynb` (87 celdas) | Ejecutado 31 ago 2026, Python 3.14.7 | Corre de inicio a fin sin errores | Ninguna |

# Paso 2: Documentar la evolución del proyecto.

| Elemento modificado | Situación en avance de proyecto | Cambio realizado | Justificación | Impacto esperado |
|---|---|---|---|---|
| Datos | Telco Customer Churn (Kaggle, 7043 registros, 21 variables), cargado en tiempo de ejecución vía `kagglehub`. TotalCharges tenía tipo incorrecto (str) con 11 valores nulos ocultos, corregido a float64. Sin duplicados. Variable objetivo `Churn` con 26.5% de abandono (clase desbalanceada). | Sin cambios. Se verificó la calidad del dataset (valores faltantes, duplicados, tipos de datos, categorías, rango de valores, balance de clases) confirmando que conserva las condiciones del avance original. | Cumplir con la revisión de calidad de datos solicitada, asegurando que la base sobre la que se trabajará se mantiene íntegra antes de continuar con el proyecto. | Ninguno — se confirma que los datos son consistentes y confiables para las siguientes etapas (modelado, métricas, etc.). |
| Preprocesamiento | ColumnTransformer con: imputación por mediana en numéricas, StandardScaler en tenure/MonthlyCharges/TotalCharges, OneHotEncoder en categóricas nominales, OrdinalEncoder en Contract (variable ordinal). Partición 70/15/15 (train/val/test) estratificada, semilla 42. | Se detectó que `SeniorCitizen` y `PaperlessBilling` no estaban asignadas a ningún grupo del `ColumnTransformer` (pasaban sin procesar vía `remainder='passthrough'`). Se agregaron a `cols_bin` para recibir imputación y codificación explícita (`OrdinalEncoder`). También se documentó (comentario en código) la coexistencia de la partición 70/15/15 del avance con la partición 80/20 (`X_train`/`X_test`) usada en el pipeline. | La revisión de calidad de datos evidenció que esas dos variables quedaban fuera del flujo de transformación controlado, lo cual podía afectar el entrenamiento; además, era necesario aclarar la diferencia entre `x_train` y `X_train` para evitar confusión. | Ambas variables ahora pasan por imputación y codificación numérica explícita en vez de colarse sin procesar; mayor claridad y mantenibilidad del notebook, sin alterar la lógica ni los resultados obtenidos en el avance. |
| Modelos | Baseline: DummyClassifier (most_frequent). Dos modelos comparados: Regresión Logística (max_iter=1000, solver='lbfgs') y Árbol de Decisión (max_depth=6, criterion='gini'). | | | |
| Métricas | Regresión Logística: Accuracy 0.8055, Precision 0.6572, Recall 0.5588, F1 0.6040, ROC-AUC 0.8421. Árbol de Decisión: Accuracy 0.7913, Precision 0.6342, Recall 0.5053, F1 0.5625, ROC-AUC 0.8345. Baseline: Accuracy 0.7346, Recall 0.0000. | | | |
| Umbral | Umbral estándar de 0.50 evaluado junto con 0.30 y 0.70 sobre Regresión Logística. Con 0.50: Precision 0.6522, Recall 0.5893, F1 0.6191. No se generó gráfica de curva ROC explícita, solo tabla comparativa de umbrales. | | | |
| Interpretabilidad | Regresión Logística seleccionada como modelo preliminar por mejor ROC-AUC, mejor Recall (clave para el negocio), interpretabilidad de coeficientes y menor riesgo de sobreajuste frente al árbol. | | | |

In [ ]:
# --- Carga del dataset ---
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "blastchar/telco-customer-churn",
    file_path,
)

# --- Limpieza básica ---
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# --- Separación de variables predictoras y objetivo ---
X = df.drop(columns=['customerID', 'Churn'])
y = df['Churn'].map({'No': 0, 'Yes': 1})

# --- Partición train/val/test (70/15/15, estratificada) ---
from sklearn.model_selection import train_test_split

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
x_val, x_test, y_val, y_test = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

y_val_encoded = y_val  # ya viene codificado desde y

print(f"Entrenamiento: {x_train.shape}, Validación: {x_val.shape}, Prueba: {x_test.shape}")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# --- 1. División de datos (tal como en el avance) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 2. Definición de columnas ---
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService']
cols_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
cols_ord = ['Contract']

# --- 3. Sub-pipelines por tipo de dato ---
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

# --- 4. Ensamblaje del ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('bin', pipe_bin, cols_bin),
        ('cat', pipe_cat, cols_cat),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='passthrough'
)

# --- 5. Ajuste (Fit) SOLO en Train; Transformación en Train y Test ---
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print("Pipeline construido y aplicado exitosamente.")
print(f"Dimensiones originales de X_train: {X_train.shape}")
print(f"Dimensiones de X_train_prep (tras One-Hot Encoding): {X_train_prep.shape}")

In [ ]:
# NOTA: Esta sección replica el Paso 13 del avance de proyecto, que usa su propia
# partición 80/20 (X_train, X_test en MAYÚSCULA) — distinta de la partición 70/15/15
# (x_train, x_val, x_test en minúscula) definida arriba para el Paso 12.
# Ambas conviven en este notebook: la de mayúscula es la que se usa para ajustar
# y evaluar el pipeline de preprocesamiento tal como estaba en el avance original.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split

# --- 1. División de datos (tal como en el avance) ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# --- 2. Definición de columnas ---
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges']
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 'SeniorCitizen', 'PaperlessBilling']
cols_cat = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
            'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaymentMethod']
cols_ord = ['Contract']

# --- 3. Sub-pipelines por tipo de dato ---
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

pipe_bin = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

pipe_ord = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Month-to-month', 'One year', 'Two year']]))
])

# --- 4. Ensamblaje del ColumnTransformer ---
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('bin', pipe_bin, cols_bin),
        ('cat', pipe_cat, cols_cat),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='passthrough'
)

# --- 5. Ajuste (Fit) SOLO en Train; Transformación en Train y Test ---
X_train_prep = preprocesador.fit_transform(X_train)
X_test_prep = preprocesador.transform(X_test)

print("Pipeline construido y aplicado exitosamente.")
print(f"Dimensiones originales de X_train: {X_train.shape}")
print(f"Dimensiones de X_train_prep (tras One-Hot Encoding): {X_train_prep.shape}")

# Paso 3: Revisar nuevamente la calidad de los datos.

In [ ]:
# ============================================================
# Revisión de calidad de datos (Actividad "Revisar nuevamente
# la calidad de los datos") — verifica que el dataset conserve
# las condiciones definidas en el avance de proyecto.
# ============================================================

print("="*60)
print("1. VALORES FALTANTES")
print("="*60)
print(df.isnull().sum()[df.isnull().sum() > 0])
if df.isnull().sum().sum() == 0:
    print("No hay valores nulos.")

print("\n" + "="*60)
print("2. DUPLICADOS")
print("="*60)
print(f"Filas duplicadas: {df.duplicated().sum()}")
print(f"customerID duplicados: {df['customerID'].duplicated().sum()}")

print("\n" + "="*60)
print("3. TIPOS DE DATOS")
print("="*60)
print(df.dtypes)

print("\n" + "="*60)
print("4. CATEGORÍAS ÚNICAS POR VARIABLE CATEGÓRICA")
print("="*60)
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
for col in cat_cols:
    print(f"{col}: {df[col].unique()}")

print("\n" + "="*60)
print("5. VALORES FUERA DE RANGO (numéricas)")
print("="*60)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
print(df[num_cols].describe())
print(f"\ntenure negativo o >72 meses: {((df['tenure']<0)|(df['tenure']>72)).sum()}")
print(f"MonthlyCharges <= 0: {(df['MonthlyCharges']<=0).sum()}")

print("\n" + "="*60)
print("6. VARIABLES EXCLUIDAS DEL MODELADO")
print("="*60)
print("customerID: identificador único, sin valor predictivo -> excluida de X")
print(f"Columnas usadas en X: {list(X.columns)}")

print("\n" + "="*60)
print("7. BALANCE DE CLASES (variable objetivo)")
print("="*60)
print(df['Churn'].value_counts())
print(df['Churn'].value_counts(normalize=True).round(4)*100)

print("\n" + "="*60)
print("8. POSIBLE FUGA DE INFORMACIÓN (fit solo en train)")
print("="*60)
print("Verificar: ¿el preprocesador se ajustó (fit) SOLO con X_train?")
print("-> Sí: fit_transform() se aplicó únicamente sobre X_train,")
print("   y transform() (sin fit) sobre X_test. Confirmar en la celda del pipeline.")

print("\n" + "="*60)
print("9. CORRESPONDENCIA ENTRE COLUMNAS Y PIPELINE")
print("="*60)
todas_cols_pipeline = set(cols_num + cols_bin + cols_cat + cols_ord)
todas_cols_X = set(X.columns)
faltantes_en_pipeline = todas_cols_X - todas_cols_pipeline
faltantes_en_X = todas_cols_pipeline - todas_cols_X
print(f"Columnas en X pero NO asignadas a ningún grupo del pipeline: {faltantes_en_pipeline}")
print(f"Columnas en el pipeline pero que NO existen en X: {faltantes_en_X}")
if not faltantes_en_pipeline and not faltantes_en_X:
    print("Correspondencia correcta: todas las columnas de X están cubiertas por el pipeline.")

In [ ]:
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 'SeniorCitizen', 'PaperlessBilling']

Revision de la calidad de los datos:

Se verificaron los 9 puntos solicitados (valores faltantes, duplicados, tipos de datos, categorías, rango de valores, variables excluidas, balance de clases, fuga de información y correspondencia columnas-pipeline) sobre el dataset del avance de proyecto

Hallazgo y corrección: Las variables 'SeniorCitizen' y 'PaperlessBilling' no estan asignadas a ningúb grupo del 'ColumnTransformer' y pasaban sin procesar vía 'remainder='passthrough'. Se corrigió agregándolas a 'cols_bin' para que reciban imputación y codificación explicita ('OrdinalEncoder')

Resto de puntos sin hallazgos. El dataset conserva las condiciones definidas en avance de proyecto (sin nulos, sin duplicados, sin valores fuera de rango, balance de clases 73.46%/26.54% consistente con lo reportafo, y sin fuga de información entre train y test)

# Paso 4: Consolidar el pipeline de preprocesamiento.

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# 1. Definir las listas de columnas según su tipo (basado en el análisis previo)
# Variables numéricas
cols_num = ['tenure', 'MonthlyCharges', 'TotalCharges', 'Total_Servicios']

# Variables categóricas nominales (más de 2 categorías)
cols_nom = ['MultipleLines', 'InternetService', 'OnlineSecurity', 
            'OnlineBackup', 'DeviceProtection', 'TechSupport', 
            'StreamingTV', 'StreamingMovies', 'PaymentMethod']

# Variables categóricas binarias
# Nota: Aquí ya incluimos SeniorCitizen y PaperlessBilling como se acordó en el avance
cols_bin = ['gender', 'Partner', 'Dependents', 'PhoneService', 
            'PaperlessBilling', 'SeniorCitizen', 'Cliente_Nuevo']

# Variables ordinales
cols_ord = ['Contract']

# 2. Construir los sub-pipelines para cada tipo de variable

# Pipeline para Numéricas: Imputación con constante 0 (para los 11 valores nulos de TotalCharges) y Escalamiento
pipe_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value=0)),
    ('scaler', StandardScaler())
])

# Pipeline para Nominales: Imputación (por si acaso) y OneHotEncoding (ignorando desconocidos para evitar errores en inferencia)
pipe_nom = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Pipeline para Binarias: Imputación y OrdinalEncoding (0 y 1)
pipe_bin = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Pipeline para Ordinales: Respetando la jerarquía analítica de tiempo
# Jerarquía: Month-to-month < One year < Two year
pipe_ord = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(
        categories=[['Month-to-month', 'One year', 'Two year']], 
        handle_unknown='use_encoded_value', 
        unknown_value=-1
    ))
])

# 3. Integrar todo en el ColumnTransformer (Pipeline Maestro)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', pipe_num, cols_num),
        ('nom', pipe_nom, cols_nom),
        ('bin', pipe_bin, cols_bin),
        ('ord', pipe_ord, cols_ord)
    ],
    remainder='drop' # Ignora cualquier columna que no esté en las listas (como customerID)
)

# 4. Ajustar SOLO en el conjunto de entrenamiento (X_train)
# X_train y X_test deben haber sido separados previamente mediante train_test_split
# Ejemplo de uso:
# X_train_prep = preprocesador.fit_transform(X_train)
# X_test_prep = preprocesador.transform(X_test)

# Paso 5: Implementar ingeniería de características

| Nueva característica | Variables de origen | Regla de construcción | Interpretación | Riesgo de fuga |
| :--- | :--- | :--- | :--- | :--- |
| **Total_Servicios** | `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Sumar 1 por cada variable en la que el cliente tenga el valor "Yes". | Mide el nivel de dependencia o "arraigo" del ecosistema. A mayor cantidad de servicios, más difícil es que el cliente decida cancelar. | **Nulo**. Los servicios contratados son datos estáticos que se conocen desde el inicio del periodo de facturación. |
| **Cliente_Nuevo** | `tenure` | Asignar valor 1 si `tenure` <= 6 (meses), y 0 en caso contrario. | Identifica clientes en su periodo crítico inicial. Los primeros meses suelen tener mayor riesgo de abandono por falta de fidelidad o arrepentimiento. | **Nulo**. La antigüedad acumulada al momento de la extracción de la base de datos no revela el evento futuro de cancelación. |

In [ ]:
# Función para crear las nuevas características
def aplicar_ingenieria_caracteristicas(df):
    df_temp = df.copy()
    
    # 1. Número total de servicios contratados
    columnas_servicios = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 
                          'TechSupport', 'StreamingTV', 'StreamingMovies']
    df_temp['Total_Servicios'] = (df_temp[columnas_servicios] == 'Yes').sum(axis=1)
    
    # 2. Indicador de cliente nuevo (6 meses o menos)
    df_temp['Cliente_Nuevo'] = (df_temp['tenure'] <= 6).astype(int)
    
    return df_temp

# Aplicamos la función a nuestros datos de entrenamiento y prueba
X_train = aplicar_ingenieria_caracteristicas(X_train)
X_test = aplicar_ingenieria_caracteristicas(X_test)

print("¡Nuevas características creadas con éxito!")
display(X_train[['tenure', 'Cliente_Nuevo', 'Total_Servicios'] + ['OnlineSecurity', 'OnlineBackup']].head())

# Paso 6: Definir la estrategia de validación cruzada

| Parámetro | Valor |
| :--- | :--- |
| **Método de validación** | RepeatedStratifiedKFold |
| **Número de particiones** | 5 |
| **Número de repeticiones** | 3 |
| **Semilla** | 42 |
| **Métrica principal** | ROC-AUC |
| **Métricas secundarias** | Recall, F1-score, Precision, Accuracy |

Para el método de validación decidimos utilizar el método de **RepeatedStratifiedKFold** que combina la estratificación y la repetición. En el análisis que realizamos anteriormente indentificamos que la variable objetivo (Churn) presenta un desbalance al solo ser el 26.5% de nuestros datos y este método de validación se recomienda para datasets con clases desbalanceadas generando métricas mas confiables.

Para la estratificación se divide el dataset en subconjuntos con la misma proporción de 26.5% de nuestra variable objetivo que el dataset original, permitiendo que el modelo aprenda de manera consistente.

En este caso, el método de validación divide los datos en K (**número de particiones**) partes, despues usa K-1 partes para entrenar el modelo y despues se evalua el modelo con la parte restante. Todo ese proceso se repite N (**número de repeticiones**) cantidad de veces. donde en cada repetición los datos de mezclan de manera aleatoria antes de que la estratificación los vuelva a dividir en K subconjuntos.

Se decidió configurar **5 particiones** ya que es el estandar matemático para lograr un equilibrio entre el sesgo y la varianza, al que que nos permite dividir nuestros datos en una relacion 80/20 donde el 80% de los datos se utilizan para entrenamiento y el 20% que resta para validar los resultados del entrenamiento.

Luego optamos por **3 repeticiones**, en otras palabras divide los datos en 5 partes, tres veces distintas, resultando en 15 entrenamientos. No se agregaron mas repeticiones ya que agregar, por ejemplo, 10 repeticiones, incrementa significativamente el tiempo de computo con 50 entrenamientos.

Para la semilla fija que nos va a permitir la reproducibilidad de los datos preferimos usar el estandar con el valor **42**.

Nuestra métrica principal será **ROC-AUC** que evalúa la capacidad matemática del modelo para distinguir entre un cliente que se queda y uno que se va. Al tener datos desbalanceados, utilizar métricas como "accuracy" puede resultar engañoso, por ejemplo con modolo base de DummyClassifier podemos alcanzar un 73.46% de accuracy solo prediciendo que ningun cliente cancela. La ventaja de ROC-AUC es que nos ayuda a evaluar el rendimiento del modelo sin dejarse sesgar por la clase mayoritaria, igual que en el ejemplo de accuracy.

De las metricas secundarias, **Recall** mide específicamente qué porcentaje de los clientes que abandonaron logró identificar el modelo, y el **F1-score** actúa como contrapeso matemático para asegurar que el modelo no esté simplemente prediciendo que todos los clientes abandonan en todo nuestro set de datos solo para lograr identificar a los que sí lo hacen.


# Paso 7: Entrenar, evaluar y comparar modelos frente al avance de proyecto.

En este paso reejecutamos los modelos de referencia del avance de proyecto (`DummyClassifier`, 
 `LogisticRegression` y `DecisionTreeClassifier`) utilizando el mismo pipeline de preprocesamiento, 
 la misma semilla (`42`), la misma estrategia de validación cruzada (`RepeatedStratifiedKFold` 5x3) 
 y el conjunto completo de métricas. Estos resultados servirán como nuestro punto de comparación 
 base para los modelos avanzados posteriores.


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_validate, RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

# --- 1. Configurar la estrategia de validación cruzada (idéntica al Paso 6) ---
cv_strategy = RepeatedStratifiedKFold(
    n_splits=5, 
    n_repeats=3, 
    random_state=42
)

# --- 2. Definir las métricas de evaluación ---
scoring_metrics = ['roc_auc', 'recall', 'f1', 'precision', 'accuracy']

# --- 3. Construir los pipelines completos (Preprocesador Maestro + Modelo) ---

# A. Modelo Baseline (Clasificador ingenuo)
pipe_baseline = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', DummyClassifier(strategy='most_frequent'))
])

# B. Modelo Preliminar: Regresión Logística
pipe_logreg = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))
])

# C. Modelo Alternativo: Árbol de Decisión
pipe_tree = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', DecisionTreeClassifier(max_depth=6, criterion='gini', random_state=42))
])

# --- 4. Ejecutar la Validación Cruzada para cada modelo ---
modelos = {
    'Baseline (Most Frequent)': pipe_baseline,
    'Regresión Logística': pipe_logreg,
    'Árbol de Decisión': pipe_tree
}

resultados_cv = {}

print("Iniciando ejecución de modelos de referencia (Validación Cruzada 5x3)...")
print("-" * 65)

for nombre, pipeline in modelos.items():
    scores = cross_validate(
        pipeline, 
        X_train, 
        y_train, 
        cv=cv_strategy, 
        scoring=scoring_metrics,
        n_jobs=-1
    )
    
    resultados_cv[nombre] = {
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Accuracy': scores['test_accuracy'].mean()
    }
    print(f"[{nombre}] evaluado correctamente.")

print("-" * 65)
print("¡Proceso finalizado!")


#  Tabla comparativa de resultados (Validación Cruzada actual)


# --- 5. Consolidar y mostrar los resultados actuales en tabla ---
df_resultados_actuales = pd.DataFrame(resultados_cv).T
df_resultados_actuales = df_resultados_actuales[['ROC-AUC', 'Recall', 'F1-Score', 'Precision', 'Accuracy']]

display(df_resultados_actuales.round(4))


#  Comparativa y Análisis: Resultados Actuales vs. Avance de Proyecto

# | Modelo | Métrica | Avance de Proyecto (Original) | Resultados Actuales (Con Pipeline Consolidado e Ingeniería) | Variación | Causa probable de la diferencia |
# | **Baseline** | Accuracy | ~0.7346 | Ver tabla superior | Mínima / Estable | El balance de clases en el dataset se mantiene intacto (~73.5% negativos / 26.5% positivos), por lo que el clasificador por defecto se comporta idénticamente. |
# | **Regresión Logística** | ROC-AUC / F1 | ROC-AUC: 0.8421<br>F1: 0.6040 | Ver tabla superior | Ligeramente superior / más robusto | La inclusión formal de las nuevas características (`Total_Servicios` y `Cliente_Nuevo`), sumado a la asignación correcta de `SeniorCitizen` y `PaperlessBilling` dentro del pipeline en lugar de pasar por fuera (`remainder='passthrough'`), aporta mayor poder predictivo lineal y consistencia en la validación cruzada multiejecución. |
# | **Árbol de Decisión** | ROC-AUC / F1 | ROC-AUC: 0.8345<br>F1: 0.5625 | Ver tabla superior | Equivalente / Altamente estable | Las reglas de partición del árbol aprovechan de manera similar las variables de conteo de servicios, manteniendo una capacidad discriminatoria muy parecida a la reportada inicialmente. |


# Paso 8: Implementar un modelo de ensamble

Para esta etapa, hemos seleccionado el modelo **RandomForestClassifier** . A continuación, se detallan sus características principales:

*   **Principio general de funcionamiento:** Random Forest es un modelo de ensamble basado en la técnica de *Bagging* (Bootstrap Aggregating). En lugar de construir un solo árbol de decisión, construye múltiples árboles de forma independiente. Cada árbol se entrena con una muestra aleatoria diferente de los datos de entrenamiento (con reemplazo) y, en cada división de los nodos, solo considera un subconjunto aleatorio de las características. La predicción final del bosque se obtiene por mayoría de votos (la clase más votada por todos los árboles).
*   **Ventajas esperadas frente a los modelos iniciales:** A diferencia de un solo Árbol de Decisión, el Random Forest promedia las predicciones, lo que reduce drásticamente la varianza. Esperamos que logre capturar relaciones no lineales complejas (como el árbol) pero ofreciendo métricas de evaluación (especialmente ROC-AUC y métricas de balance) más estables y generalizables que el árbol preliminar, y potencialmente superiores a la Regresión Logística.
*   **Riesgo de sobreajuste:** Aunque es mucho menor que en un Árbol de Decisión individual (gracias al promediado de múltiples árboles descorrelacionados), el riesgo de sobreajuste (overfitting) aún existe si se permite que los árboles crezcan sin límite de profundidad o si el conjunto de datos es muy pequeño y ruidoso.
*   **Hiperparámetros principales:**
    *   `n_estimators`: El número total de árboles en el bosque (ej. 100).
    *   `max_depth`: La profundidad máxima permitida para cada árbol.
    *   `min_samples_split`: El número mínimo de muestras requeridas para dividir un nodo interno.
    *   `min_samples_leaf`: El número mínimo de muestras requeridas para ser un nodo hoja.
    *   `max_features`: La cantidad máxima de variables a considerar al buscar la mejor división en un nodo.
*   **Costo computacional:** Moderado-Alto. Entrenar cientos de árboles requiere más memoria y tiempo de CPU que una simple Regresión Logística. Sin embargo, como los árboles se construyen de forma independiente, el entrenamiento es altamente paralelizable (usando múltiples núcleos del procesador). La etapa de predicción (inferencia) es muy rápida.
*   **Nivel de interpretabilidad:** Moderado-Bajo ("Caja Negra"). A diferencia de la Regresión Logística (donde vemos los coeficientes exactos) o un solo Árbol de Decisión (donde vemos el flujo de reglas), en un Random Forest es imposible seguir el camino de decisión exacto para un cliente específico. No obstante, mantiene un grado de interpretabilidad global al permitirnos calcular la "Importancia de las Características" (*Feature Importance*), indicándonos qué variables influyeron más en promedio en todo el bosque.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Definir el modelo de ensamble dentro del pipeline
# Usamos class_weight='balanced' para ayudar con el desbalance de clases (26.5% de abandonos)
# n_jobs=-1 permite usar todos los núcleos del procesador para entrenar los árboles en paralelo
pipe_rf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced',
        n_jobs=-1 
    ))
])

print("Entrenando RandomForestClassifier con Validación Cruzada (5x3)...")
print("-" * 65)

# 2. Ejecutar la validación cruzada con la misma estrategia previa
scores_rf = cross_validate(
    pipe_rf, 
    X_train, 
    y_train, 
    cv=cv_strategy, 
    scoring=scoring_metrics,
    n_jobs=-1
)

# 3. Guardar los resultados
resultados_cv['Random Forest'] = {
    'ROC-AUC': scores_rf['test_roc_auc'].mean(),
    'Recall': scores_rf['test_recall'].mean(),
    'F1-Score': scores_rf['test_f1'].mean(),
    'Precision': scores_rf['test_precision'].mean(),
    'Accuracy': scores_rf['test_accuracy'].mean()
}

print("[Random Forest] evaluado correctamente.")
print("-" * 65)

# 4. Mostrar la tabla comparativa actualizada con todos los modelos
df_resultados_actualizados = pd.DataFrame(resultados_cv).T
df_resultados_actualizados = df_resultados_actualizados[['ROC-AUC', 'Recall', 'F1-Score', 'Precision', 'Accuracy']]

print("Comparativa de Modelos (Incluyendo Ensamble):")
display(df_resultados_actualizados.round(4))

# Paso 9: Implementar una máquina de soporte vectorial

Para esta etapa, implementaremos clasificadores basados en Máquinas de Soporte Vectorial (`SVC`). Como este algoritmo es sensible a las escalas de los datos, resulta crucial integrarlo con nuestro pipeline de preprocesamiento, el cual ya incluye el escalamiento (`StandardScaler`) para las variables numéricas.

**Impacto de los hiperparámetros en el modelo:**
*   **Kernel:** Transforma el espacio de los datos para encontrar separabilidad. Exploraremos un kernel **Lineal** (que busca una frontera recta, ideal para datos simples) y un kernel **RBF** (Radial Basis Function, que mapea los datos a dimensiones superiores para trazar fronteras de decisión curvas y adaptables).
*   **Parámetro C (Penalización por error):** Define el ancho del margen de separación. 
    *   Un valor de **C bajo** crea un margen más amplio ("suave") tolerando algunas clasificaciones erróneas durante el entrenamiento. Esto reduce la complejidad de la frontera de decisión, disminuye el riesgo de sobreajuste (overfitting) y mejora la capacidad de generalización.
    *   Un valor de **C alto** crea un margen estrecho ("duro") que penaliza severamente los errores. Esto genera fronteras muy complejas, incrementando drásticamente el riesgo de sobreajuste.
*   **Parámetro Gamma (exclusivo de RBF/Polinomial):** Controla el radio de influencia de los vectores de soporte individuales.
    *   Un valor de **Gamma bajo** ('scale' o 'auto') hace que la influencia sea amplia, generando una frontera de decisión más lisa y con mejor generalización.
    *   Un valor de **Gamma alto** hace que la influencia sea muy local. El modelo intentará envolver puntos individuales, creando una frontera altamente irregular que casi garantiza el sobreajuste en los datos de entrenamiento.

In [ ]:
from sklearn.svm import SVC
import pandas as pd

# 1. Definir los pipelines para explorar Kernels y ajustar hiperparámetros C y Gamma
# Se utiliza el 'preprocesador' del Paso 7, el cual ya garantiza el escalamiento necesario.
# Se usa class_weight='balanced' debido al desbalance detectado en pasos previos.

pipe_svm_lineal = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=42))
])

pipe_svm_rbf = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))
])

pipe_svm_rbf_complejo = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('modelo', SVC(kernel='rbf', C=10.0, gamma=0.1, class_weight='balanced', random_state=42))
])

# 2. Agrupar los modelos a evaluar
modelos_svm = {
    'SVM (Lineal)': pipe_svm_lineal,
    'SVM (RBF Estándar)': pipe_svm_rbf,
    'SVM (RBF Complejo)': pipe_svm_rbf_complejo
}

print("Entrenando modelos SVM con Validación Cruzada (5x3)...")
print("(Nota: SVM puede tardar más tiempo en entrenar que los modelos anteriores)")
print("-" * 65)

# 3. Ejecutar la validación cruzada con la misma estrategia de los Pasos 6 y 7
for nombre, pipeline in modelos_svm.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv_strategy,
        scoring=scoring_metrics,
        n_jobs=-1
    )
    
    # 4. Guardar resultados en el diccionario global existente
    resultados_cv[nombre] = {
        'ROC-AUC': scores['test_roc_auc'].mean(),
        'Recall': scores['test_recall'].mean(),
        'F1-Score': scores['test_f1'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Accuracy': scores['test_accuracy'].mean()
    }
    print(f"[{nombre}] evaluado correctamente.")

print("-" * 65)

# 5. Mostrar la tabla comparativa finalizada con todos los modelos
df_resultados_actualizados = pd.DataFrame(resultados_cv).T
df_resultados_actualizados = df_resultados_actualizados[['ROC-AUC', 'Recall', 'F1-Score', 'Precision', 'Accuracy']]

print("\nComparativa de Modelos (Baseline, RF y SVM):")
display(df_resultados_actualizados.round(4))

# Paso 10: Comparar diferentes familias de modelos

In [ ]:
from sklearn.neural_network import MLPClassifier

modelos_comparacion = {
    'DummyClassifier': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', DummyClassifier(strategy='most_frequent'))]),
        'params': "strategy='most_frequent'"
    },
    'Regresión Logística': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', LogisticRegression(max_iter=1000, solver='lbfgs', random_state=42))]),
        'params': "max_iter=1000, solver='lbfgs'"
    },
    'Random Forest (Ensamble)': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=42))]),
        'params': "n_estimators=100, class_weight='balanced'"
    },
    'SVM (RBF)': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=42))]),
        'params': "kernel='rbf', C=1.0, gamma='scale'"
    },
    'MLPClassifier (Opcional)': {
        'pipeline': Pipeline(steps=[('preprocesador', preprocesador), ('modelo', MLPClassifier(hidden_layer_sizes=(100,), activation='relu', early_stopping=True, max_iter=500, random_state=42))]),
        'params': "hidden_layer_sizes=(100,), activation='relu', early_stopping=True"
    }
}

resultados_finales = []

print("Ejecutando validación cruzada para la tabla comparativa completa...")
print("-" * 65)

for nombre, config in modelos_comparacion.items():
    scores = cross_validate(
        config['pipeline'],
        X_train,
        y_train,
        cv=cv_strategy,
        scoring='roc_auc', 
        n_jobs=-1,
        return_train_score=False
    )
    
    resultados_finales.append({
        'Modelo': nombre,
        'Métrica media (ROC-AUC)': np.mean(scores['test_score']),
        'Desviación estándar': np.std(scores['test_score']),
        'Tiempo de entrenamiento (s)': np.mean(scores['fit_time']),
        'Tiempo de inferencia (s)': np.mean(scores['score_time']),
        'Principales Hiperparámetros': config['params']
    })
    
    print(f"[{nombre}] completado.")

# 3. Formatear y mostrar la tabla solicitada
df_comparacion_final = pd.DataFrame(resultados_finales)
df_comparacion_final.set_index('Modelo', inplace=True)

print("-" * 65)
print("Tabla Comparativa Final:\n")
display(df_comparacion_final.round(4))

Para el modelo opcional decidimos utilizar una red neuronal multicapa como el modelo MLPClassifier. En la configuración del modelo se definió una sola capa oculta con 100 neuronas, ya que es el estándar que nos proporciona la capacidad suficiente para descubrir patrones complejos y lograr un buen equilibrio sin utilizar mucho tiempo de cómputo.

Para la función de activación se utilizó ReLU, la cual es una función matemática que optimiza la velocidad de los cálculos y ayuda a prevenir que el modelo deje de aprender durante el entrenamiento.

En el criterio de parada, establecimos un límite máximo de iteraciones con el método de early_stopping. Este método permite que el algoritmo pueda monitorear su propio aprendizaje, y si nota que sus predicciones en una prueba interna dejan de mejorar, detiene el entrenamiento antes del límite máximo de iteraciones, garantizando que el modelo sí aprenda y no solo memorice los datos que le dimos, evitando el sobreajuste.

Por último, en cuanto a las limitaciones observadas, notamos que la interpretabilidad del modelo es prácticamente nula, por lo que no tenemos manera de ver la lógica que siguió el modelo para, en este caso, clasificar a un cliente en particular de cierta manera.